<a href="https://colab.research.google.com/github/digitaldaimyo/AddressedStateAttention/blob/main/notebooks/asa_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Addressed State Attention (ASA)

A research harness for **Addressed State Attention** language models: inspect slot routing, run causal slot masking, and probe refine-delta geometry — all from a single checkpoint.

**Quick links:** [Hugging Face checkpoint](https://huggingface.co/DigitalDaimyo/AddressedStateAttention) · [FineWeb 187M @75k](https://huggingface.co/DigitalDaimyo/AddressedStateAttention/blob/main/checkpoints/fineweb_187M_75k.pt) · [GitHub repo](https://github.com/DigitalDaimyo/AddressedStateAttention)

## Setup

In [ ]:

#@title Install + Imports
!pip install -q git+https://github.com/DigitalDaimyo/AddressedStateAttention.git \
  huggingface_hub transformers

import os
import copy
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

from asa import load_asm_checkpoint, generate

SEED = 0
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:

#@title Load checkpoint from Hugging Face
ckpt_path = hf_hub_download(
    repo_id="DigitalDaimyo/AddressedStateAttention",
    filename="checkpoints/fineweb_187M_75k.pt",
)
print("ckpt_path:", ckpt_path)

model, cfg, ckpt = load_asm_checkpoint(ckpt_path, mode="analysis")
model = model.to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained("gpt2")
print("loaded ✓")

In [ ]:

#@title Quick generation sanity check
prompt = "John knew what he had to"

with torch.no_grad():
    out = generate(
        model,
        tokenizer,
        prompt,
        max_new_tokens=120,
        strategy="sample",
        temperature=0.75,
        top_p=0.92,
        top_k=50,
        repetition_penalty=1.2,
        no_repeat_ngram_size=4,
        device=device,
    )

print("Prompt:", repr(prompt))
print("Gen:", out)

## Analysis Tools

In [ ]:

#@title Harness controls
info_cfg = dict(
    store_read_weights=True,
    store_read_logits=True,
    store_write_logits=False,
    store_slot_state_norm=True,
    store_out1=True,
    store_delta=True,
    store_slot_w=True,
    detach_to_cpu=True,
    time_stride=1,
    batch_stride=1,
)
INFO_LEVEL = "full"

In [ ]:

#@title Helpers — reset + context manager for safe interventions
def iter_asa_modules(model):
    for blk in getattr(model, "blocks", []):
        asa = getattr(blk, "asa", None)
        if asa is not None:
            yield asa

_ASA_STICKY_ATTRS = [
    "routing_override",
    "slot_mask",
    "_intv_mode", "_log_refine_geom",
    "_intv_beta", "_intv_par_beta",
    "_intv_score_kind", "_intv_tau_kind", "_intv_tau_pctl", "_intv_tau",
    "_intv_mask_mode", "_intv_soft_temp",
    "_intv_score_clip_pctl",
    "_intv_head_mask",
]

def snapshot_asa_state(asa):
    snap = {}
    for name in _ASA_STICKY_ATTRS:
        if hasattr(asa, name):
            val = getattr(asa, name)
            snap[name] = val.detach().clone() if torch.is_tensor(val) else copy.deepcopy(val)
    return snap

def restore_asa_state(asa, snap):
    for name, val in snap.items():
        setattr(asa, name, val)

def reset_asa_to_defaults(asa):
    if hasattr(asa, "routing_override"):
        asa.routing_override = None
    if hasattr(asa, "slot_mask"):
        asa.slot_mask = None

    if hasattr(asa, "_intv_mode"):
        asa._intv_mode = "off"
    if hasattr(asa, "_log_refine_geom"):
        asa._log_refine_geom = False
    if hasattr(asa, "_intv_beta"):
        asa._intv_beta = 1.0
    if hasattr(asa, "_intv_par_beta"):
        asa._intv_par_beta = 1.0
    if hasattr(asa, "_intv_score_kind"):
        asa._intv_score_kind = "orth_frac"
    if hasattr(asa, "_intv_tau_kind"):
        asa._intv_tau_kind = "pctl"
    if hasattr(asa, "_intv_tau_pctl"):
        asa._intv_tau_pctl = 75.0
    if hasattr(asa, "_intv_tau"):
        asa._intv_tau = 0.0
    if hasattr(asa, "_intv_mask_mode"):
        asa._intv_mask_mode = "soft"
    if hasattr(asa, "_intv_soft_temp"):
        asa._intv_soft_temp = 0.05
    if hasattr(asa, "_intv_score_clip_pctl"):
        asa._intv_score_clip_pctl = 99.0
    if hasattr(asa, "_intv_head_mask"):
        asa._intv_head_mask = None

def reset_model(model, *, device=None, set_eval=True, clear_cuda_cache=False, verbose=False):
    n = 0
    for asa in iter_asa_modules(model):
        reset_asa_to_defaults(asa)
        n += 1
    if set_eval:
        model.eval()
    if device is not None:
        model.to(device)
    if clear_cuda_cache and torch.cuda.is_available():
        torch.cuda.empty_cache()
    if verbose:
        print(f"reset_model ✓ (reset {n} ASA modules)")

@contextmanager
def intervention(model, *, device=None, force_eval=True, reset_before=True, clear_cuda_cache=False):
    was_training = model.training
    asas = list(iter_asa_modules(model))
    snaps = [snapshot_asa_state(asa) for asa in asas]
    try:
        if reset_before:
            reset_model(model, device=device, set_eval=force_eval, clear_cuda_cache=clear_cuda_cache, verbose=False)
        else:
            if force_eval: model.eval()
            if device is not None: model.to(device)
        yield
    finally:
        for asa, snap in zip(asas, snaps):
            restore_asa_state(asa, snap)
        model.train(was_training)

In [ ]:

#@title Helpers — forward_with_info + distributions + metrics
@torch.no_grad()
def forward_with_info(
    text: str,
    *,
    routing_mode="softmax",
    routing_topk=2,
    slot_mask=None,
    slot_mask_where="read",
    slot_mask_scope="all",
):
    enc = tokenizer(text, return_tensors="pt")
    input_ids = enc.input_ids.to(device)
    attn_mask = enc.attention_mask.to(device) if "attention_mask" in enc else None
    logits, infos = model(
        input_ids,
        attention_mask=attn_mask,
        return_info=True,
        routing_mode=routing_mode,
        routing_topk=routing_topk,
        slot_mask=slot_mask,
        slot_mask_where=slot_mask_where,
        slot_mask_scope=slot_mask_scope,
        info_level=INFO_LEVEL,
        info_cfg=info_cfg,
    )
    return logits, infos, input_ids

@torch.no_grad()
def probs_last_token(logits):  # [V]
    return F.softmax(logits[0, -1, :], dim=-1)

@torch.no_grad()
def topk_table(probs, k=12):
    vals, idx = torch.topk(probs, k=k)
    toks = [tokenizer.decode([i]).replace("\n", "\\n") for i in idx.tolist()]
    return pd.DataFrame({"token": toks, "prob": vals.cpu().numpy(), "id": idx.cpu().numpy()})

@torch.no_grad()
def js_divergence(p, q, eps=1e-12):
    p = (p + eps) / (p.sum() + eps)
    q = (q + eps) / (q.sum() + eps)
    m = 0.5 * (p + q)
    return float((0.5 * (torch.sum(p * torch.log(p / m)) + torch.sum(q * torch.log(q / m)))).item())

## Introductory Analysis

In [ ]:

#@title WRITE — causal slot masking (safe + tables + plots)
prompt = "John knew what he had to"

with intervention(model, device=device, reset_before=True):
    logits0, infos0, _ = forward_with_info(prompt)

rw = infos0[-1]["read_weights"][0, :, -1, :]  # [H,K]
rw_mean = rw.mean(dim=0)
top_slots = torch.topk(rw_mean, k=4).indices
K = rw_mean.numel()

slot_mask = torch.ones(K, dtype=torch.float32, device=device)
slot_mask[top_slots] = 0.0

with intervention(model, device=device, reset_before=True):
    logits1, infos1, _ = forward_with_info(
        prompt,
        slot_mask=slot_mask,
        slot_mask_where="read",
        slot_mask_scope="last_pos_only",
    )

p0 = probs_last_token(logits0).cpu()
p1 = probs_last_token(logits1).cpu()

df0 = topk_table(p0, k=12)
df1 = topk_table(p1, k=12)
js = js_divergence(p0, p1)

print("Masked slots:", top_slots.cpu().tolist())
print(f"JS divergence: {js:.6f}\n")

print("Baseline top next tokens:")
display(df0[["token", "prob"]])

print("Masked top next tokens:")
display(df1[["token", "prob"]])

ids = sorted(set(df0["id"].tolist() + df1["id"].tolist()))
toks = [tokenizer.decode([i]).replace("\n", "\\n") for i in ids]
delta = (p1[ids] - p0[ids]).numpy()
df_delta = pd.DataFrame({"token": toks, "delta": delta}).sort_values("delta", ascending=False).head(20)
display(df_delta)

fig = plt.figure(figsize=(16, 4))
ax1 = fig.add_subplot(1, 3, 1)
ax1.bar(np.arange(K), rw_mean.cpu().numpy())
for s in top_slots.cpu().tolist():
    ax1.axvline(s, linestyle="--", linewidth=1)
ax1.set_title("Mean read weight per slot\n(masked slots dashed)")
ax1.set_xlabel("slot")

ax2 = fig.add_subplot(1, 3, 2)
ax2.imshow(rw.cpu().numpy(), aspect="auto")
ax2.set_title("Read weights by head × slot\n(last token)")
ax2.set_xlabel("slot")
ax2.set_ylabel("head")

ax3 = fig.add_subplot(1, 3, 3)
ax3.bar(np.arange(len(df_delta)), df_delta["delta"].to_numpy())
ax3.set_title("Δ next-token probability\n(masked − baseline), top 20")
ax3.set_xticks(np.arange(len(df_delta)))
ax3.set_xticklabels(df_delta["token"].tolist(), rotation=60, ha="right")
ax3.set_ylabel("Δ prob")

plt.tight_layout()
plt.show()

In [ ]:

#@title READ — slot usage at last position (safe + tables + plots)
prompt = "John knew what he had to"

with intervention(model, device=device, reset_before=True):
    logits, infos, input_ids = forward_with_info(prompt)

info = infos[-1]
rw = info["read_weights"]
if rw is None:
    raise RuntimeError("read_weights is None. Ensure info_cfg['store_read_weights']=True.")

rw_last = rw[0, :, -1, :]      # [H,K]
rw_mean = rw_last.mean(dim=0)  # [K]
H, K = rw_last.shape

df_top = pd.DataFrame({
    "slot": torch.topk(rw_mean, k=min(12, K)).indices.cpu().numpy().astype(int),
    "mean_read_weight": torch.topk(rw_mean, k=min(12, K)).values.cpu().numpy(),
}).sort_values("mean_read_weight", ascending=False).reset_index(drop=True)

print(f"Prompt: {prompt!r}")
print(f"Layer: -1 | Heads: {H} | Slots: {K} | SeqLen: {rw.shape[2]}")
display(df_top)

fig = plt.figure(figsize=(16, 4))
ax1 = fig.add_subplot(1, 3, 1)
ax1.bar(np.arange(K), rw_mean.cpu().numpy())
ax1.set_title("Mean read weight per slot\n(last token, mean over heads)")
ax1.set_xlabel("slot")
ax1.set_ylabel("mean weight")

ax2 = fig.add_subplot(1, 3, 2)
ax2.imshow(rw_last.cpu().numpy(), aspect="auto")
ax2.set_title("Read weights by head × slot\n(last token)")
ax2.set_xlabel("slot")
ax2.set_ylabel("head")
ax2.set_xticks(np.arange(K))
ax2.set_yticks(np.arange(H))

ax3 = fig.add_subplot(1, 3, 3)
p = probs_last_token(logits)
dfp = topk_table(p, k=12)
ax3.bar(np.arange(len(dfp)), dfp["prob"].to_numpy())
ax3.set_title("Top next-token probabilities\n(model output sanity check)")
ax3.set_xticks(np.arange(len(dfp)))
ax3.set_xticklabels(dfp["token"].tolist(), rotation=60, ha="right")
ax3.set_ylabel("prob")

plt.tight_layout()
plt.show()

In [ ]:

#@title REFINE — compare off vs delta_orth (safe + tables + plots)
def configure_refine(mode="off", *, log_geom=True):
    for asa in iter_asa_modules(model):
        asa._intv_mode = str(mode)
        asa._log_refine_geom = bool(log_geom)

prompt = "John knew what he had to"

with intervention(model, device=device, reset_before=True):
    configure_refine("off", log_geom=True)
    logits0, infos0, _ = forward_with_info(prompt)

with intervention(model, device=device, reset_before=True):
    configure_refine("delta_orth", log_geom=True)
    logits1, infos1, _ = forward_with_info(prompt)

p0 = probs_last_token(logits0).cpu()
p1 = probs_last_token(logits1).cpu()

df0 = topk_table(p0, k=8)
df1 = topk_table(p1, k=8)
js = js_divergence(p0, p1)

print("Geom keys:", [k for k in infos1[-1].keys() if k.startswith("geom_")])
g = infos1[-1].get("geom_orth_frac", None)
if g is not None:
    print("\nLast-layer geom_orth_frac (per head):")
    print(g)

print("\nBaseline top next tokens:")
display(df0[["token", "prob"]])

print("\nDelta-orth top next tokens:")
display(df1[["token", "prob"]])

fig = plt.figure(figsize=(16, 4))

ax1 = fig.add_subplot(1, 3, 1)
ax1.bar([0, 1], [0.0, js])
ax1.set_xticks([0, 1])
ax1.set_xticklabels(["off", "delta_orth"])
ax1.set_title("Distribution shift vs off\n(JS divergence)")
ax1.set_ylabel("JS")

ax2 = fig.add_subplot(1, 3, 2)
if g is not None:
    ax2.plot(g.detach().cpu().numpy())
    ax2.set_title("geom_orth_frac per head\n(last layer)")
    ax2.set_xlabel("head")
    ax2.set_ylabel("orth_frac")
else:
    ax2.text(0.5, 0.5, "geom_orth_frac not present", ha="center", va="center")
    ax2.set_axis_off()

ax3 = fig.add_subplot(1, 3, 3)
k_union = 20
ids = sorted(set(torch.topk(p0, k_union).indices.tolist() + torch.topk(p1, k_union).indices.tolist()))
toks = [tokenizer.decode([i]).replace("\n", "\\n") for i in ids]
delta = (p1[ids] - p0[ids]).numpy()
order = np.argsort(-delta)
toks = [toks[i] for i in order]
delta = delta[order]
ax3.bar(np.arange(len(ids)), delta)
ax3.set_title("Δ prob (delta_orth − off)\nunion top-20")
ax3.set_xticks(np.arange(len(ids)))
ax3.set_xticklabels(toks, rotation=60, ha="right")
ax3.set_ylabel("Δ prob")

plt.tight_layout()
plt.show()

## Mapping Behaviors: Summary Stats and Plots Across Heads and Layers

In [ ]:

#@title Deep Dive Toolkit — summary stats + plotting utilities
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

def _to_numpy(x):
    if x is None:
        return None
    if torch.is_tensor(x):
        return x.detach().float().cpu().numpy()
    return np.asarray(x)

@torch.no_grad()
def routing_stats_from_read_weights(rw_bhTk: torch.Tensor, *, eps=1e-12):
    """
    rw_bhTk: [B,H,T,K] read weights (probabilities over slots)
    Returns dict of stats aggregated over B,T: arrays [H] or scalars.
    """
    # ensure float
    rw = rw_bhTk.float()

    # entropy per token per head: H = -sum p log p
    ent = -(rw * (rw + eps).log()).sum(dim=-1)              # [B,H,T]
    ent_mean = ent.mean(dim=(0, 2))                         # [H]

    # effective slots = exp(entropy)
    eff = torch.exp(ent)                                    # [B,H,T]
    eff_mean = eff.mean(dim=(0, 2))                         # [H]

    # top-1 mass and top-2 mass
    top1 = rw.max(dim=-1).values                            # [B,H,T]
    top2 = torch.topk(rw, k=min(2, rw.shape[-1]), dim=-1).values.sum(dim=-1)  # [B,H,T]
    top1_mean = top1.mean(dim=(0, 2))                       # [H]
    top2_mean = top2.mean(dim=(0, 2))                       # [H]

    # slot usage (mean over B,T): [H,K]
    slot_mean = rw.mean(dim=(0, 2))                         # [H,K]

    return dict(
        entropy=ent_mean,
        eff_slots=eff_mean,
        top1=top1_mean,
        top2=top2_mean,
        slot_mean=slot_mean,
    )

def layer_head_dataframe(layer_stats, *, layer_idx: int):
    """
    layer_stats: output of routing_stats_from_read_weights
    returns df with rows=heads.
    """
    H = layer_stats["entropy"].numel()
    df = pd.DataFrame({
        "layer": [layer_idx]*H,
        "head": list(range(H)),
        "entropy": _to_numpy(layer_stats["entropy"]),
        "eff_slots": _to_numpy(layer_stats["eff_slots"]),
        "top1": _to_numpy(layer_stats["top1"]),
        "top2": _to_numpy(layer_stats["top2"]),
    })
    return df

def plot_heatmap(mat, title, xlabel, ylabel, xticks=None, yticks=None):
    plt.imshow(mat, aspect="auto")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    if xticks is not None:
        plt.xticks(np.arange(len(xticks)), xticks, rotation=60, ha="right")
    if yticks is not None:
        plt.yticks(np.arange(len(yticks)), yticks)

@torch.no_grad()
def next_token_distribution(logits):
    return F.softmax(logits[0, -1, :], dim=-1)

@torch.no_grad()
def js_divergence(p, q, eps=1e-12):
    p = (p + eps) / (p.sum() + eps)
    q = (q + eps) / (q.sum() + eps)
    m = 0.5 * (p + q)
    return float((0.5 * (torch.sum(p * torch.log(p / m)) + torch.sum(q * torch.log(q / m)))).item())

@torch.no_grad()
def topk_overlap(p, q, k=50):
    p_idx = torch.topk(p, k=k).indices.tolist()
    q_idx = torch.topk(q, k=k).indices.tolist()
    return len(set(p_idx).intersection(set(q_idx))) / k

def geom_df_from_info(info: dict, layer_idx: int):
    keys = ["geom_alpha_mean","geom_alpha_abs","geom_sign_pos","geom_orth_frac","geom_d_ratio","geom_dpar_ratio"]
    keys = [k for k in keys if (k in info and info[k] is not None)]
    if not keys:
        return pd.DataFrame()
    H = int(info[keys[0]].numel())
    data = {"layer": [layer_idx]*H, "head": list(range(H))}
    for k in keys:
        data[k.replace("geom_","")] = _to_numpy(info[k])
    return pd.DataFrame(data)

In [ ]:

#@title Deep Dive (WRITE) — slot masking sweep across layers
prompt = "John knew what he had to"

mask_counts = [1, 2, 4, 8]  # how many top slots to knock out
layers_to_probe = list(range(len(model.blocks)))  # all layers; change to [0, -1] etc.

results = []

# Baseline distribution once
with intervention(model, device=device, reset_before=True):
    logits_base, infos_base, _ = forward_with_info(prompt)

p_base = next_token_distribution(logits_base).cpu()

for li in layers_to_probe:
    # determine top slots at that layer (using last token, mean over heads)
    rw = infos_base[li]["read_weights"][0, :, -1, :]  # [H,K]
    rw_mean = rw.mean(dim=0)                          # [K]
    K = rw_mean.numel()

    order = torch.argsort(rw_mean, descending=True)

    for m in mask_counts:
        m = min(m, K)
        slots = order[:m]
        slot_mask = torch.ones(K, dtype=torch.float32, device=device)
        slot_mask[slots] = 0.0

        with intervention(model, device=device, reset_before=True):
            logits_mask, infos_mask, _ = forward_with_info(
                prompt,
                slot_mask=slot_mask,
                slot_mask_where="read",
                slot_mask_scope="last_pos_only",
            )

        p_mask = next_token_distribution(logits_mask).cpu()
        results.append({
            "layer": li,
            "masked_n": int(m),
            "masked_slots": slots.cpu().tolist(),
            "js": js_divergence(p_base, p_mask),
            "top50_overlap": topk_overlap(p_base, p_mask, k=50),
        })

df = pd.DataFrame(results)
print("Masking sweep results:")
display(df.head(20))

# Plot: JS divergence vs layer for each masked_n
plt.figure(figsize=(12, 4))
for m in mask_counts:
    sub = df[df["masked_n"] == m].sort_values("layer")
    plt.plot(sub["layer"].to_numpy(), sub["js"].to_numpy(), label=f"mask {m}")
plt.title("Next-token distribution shift vs layer\n(JS divergence after masking top-N slots)")
plt.xlabel("layer")
plt.ylabel("JS divergence")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

#@title Deep Dive (READ) — routing stats across layers/heads
prompt = "John knew what he had to"

with intervention(model, device=device, reset_before=True):
    logits, infos, _ = forward_with_info(prompt)

L = len(infos)

rows = []
eff_layer_head = []
top1_layer_head = []
slot_usage_layer = []  # [L,K] mean over heads

for li in range(L):
    info = infos[li]
    rw = info.get("read_weights", None)
    if rw is None:
        raise RuntimeError("Need read_weights; set info_cfg['store_read_weights']=True.")

    st = routing_stats_from_read_weights(rw)          # dict with [H] and [H,K]
    df_lh = layer_head_dataframe(st, layer_idx=li)
    rows.append(df_lh)

    eff_layer_head.append(_to_numpy(st["eff_slots"])) # [H]
    top1_layer_head.append(_to_numpy(st["top1"]))     # [H]

    # mean slot usage over heads: [K]
    slot_mean_hk = st["slot_mean"]                    # [H,K]
    slot_usage_layer.append(_to_numpy(slot_mean_hk.mean(dim=0)))

df_lh_all = pd.concat(rows, ignore_index=True)

# Layer summary table
df_layer = df_lh_all.groupby("layer").agg({
    "entropy": "mean",
    "eff_slots": "mean",
    "top1": "mean",
    "top2": "mean",
}).reset_index()

print("Layer summary (mean over heads):")
display(df_layer)

eff_layer_head = np.stack(eff_layer_head, axis=0)     # [L,H]
top1_layer_head = np.stack(top1_layer_head, axis=0)   # [L,H]
slot_usage_layer = np.stack(slot_usage_layer, axis=0) # [L,K]

plt.figure(figsize=(16, 4))
plt.subplot(1, 3, 1)
plot_heatmap(eff_layer_head, "Effective slots (exp entropy)\nlayer × head", "head", "layer")
plt.subplot(1, 3, 2)
plot_heatmap(top1_layer_head, "Top-1 routing mass\nlayer × head", "head", "layer")
plt.subplot(1, 3, 3)
plot_heatmap(slot_usage_layer, "Mean slot usage (over heads)\nlayer × slot", "slot", "layer")
plt.tight_layout()
plt.show()

In [ ]:

#@title Deep Dive (REFINE) — geometry across layers/heads + intervention comparison
def configure_refine(mode="off", *, log_geom=True):
    for asa in iter_asa_modules(model):
        asa._intv_mode = str(mode)
        asa._log_refine_geom = bool(log_geom)

prompt = "John knew what he had to"
modes = ["off", "delta_par", "delta_orth"]  # add "orth_gate" once you're happy

mode_data = {}

for mode in modes:
    with intervention(model, device=device, reset_before=True):
        configure_refine(mode, log_geom=True)
        logits, infos, _ = forward_with_info(prompt)

    p = next_token_distribution(logits).cpu()
    mode_data[mode] = dict(logits=logits, infos=infos, probs=p)

# Metrics vs off
p_off = mode_data["off"]["probs"]
rows = []
for mode in modes:
    p = mode_data[mode]["probs"]
    rows.append({
        "mode": mode,
        "js_vs_off": js_divergence(p_off, p),
        "top50_overlap_vs_off": topk_overlap(p_off, p, k=50),
    })
dfm = pd.DataFrame(rows)
print("Intervention metrics (global):")
display(dfm)

# Geometry tables across layers/heads
geom_all = []
for mode in modes:
    infos = mode_data[mode]["infos"]
    for li, info in enumerate(infos):
        gdf = geom_df_from_info(info, layer_idx=li)
        if not gdf.empty:
            gdf["mode"] = mode
            geom_all.append(gdf)

if geom_all:
    geom = pd.concat(geom_all, ignore_index=True)
    # Focus plot: orth_frac layer×head for each mode
    for mode in modes:
        sub = geom[geom["mode"] == mode]
        if "orth_frac" not in sub.columns:
            continue
        pivot = sub.pivot(index="layer", columns="head", values="orth_frac").to_numpy()
        plt.figure(figsize=(12, 3))
        plot_heatmap(pivot, f"geom_orth_frac (layer × head) — mode={mode}", "head", "layer")
        plt.tight_layout()
        plt.show()
else:
    print("No geom_* keys found. (Ensure log_geom=True and your harness populates geom logs.)")